# EMOTIC — Ablation study: ContextEncoder vs BodyEncoder vs complete model

| Model | Input | Pipe |
|--------|---------|-------------|
| **ContextOnly** | full image | ResNet50 Places365 → Linear(256,7) |
| **BodyOnly** | body crop (Haar cascade) | ResNet50 ImageNet → Linear(256,7) |
| **CCIM completo** | full image + body crop | Both encoders + merge + CCIM |

## 1. Imports & Config

In [ ]:
from pathlib import Path

from dual_resnet_shared import (
    device, EMOTIC_LABELS,
    make_emotic_loaders,
)
from ablation_models import (
    ContextOnlyModel, BodyOnlyModel,
    train_ablation_model, evaluate_ablation_model
)

import pandas as pd
import torch

CKPT_DIR = Path("checkpoints")
ABLATION_CKPT_DIR = CKPT_DIR / "ablation"
ABLATION_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Ablation checkpoints: {ABLATION_CKPT_DIR.resolve()}")

## 2. Load data

In [ ]:
train_loader, val_loader, test_loader, class_weights = make_emotic_loaders()
print(f"Loaders: train={len(train_loader.dataset)} | val={len(val_loader.dataset)} | test={len(test_loader.dataset)}")

## 4. ContextOnly

### Training

In [ ]:
ctx_model = ContextOnlyModel(n_classes=len(EMOTIC_LABELS), freeze_backbone=True).to(device)

# Avoid re-execution
if (ABLATION_CKPT_DIR / "emotic_context_only_best.pt").exists():
    ctx_model.load_state_dict(torch.load(
        ABLATION_CKPT_DIR / "emotic_context_only_best.pt",
        map_location=device, weights_only=True,
    ))
    print(f"ContextOnly loaded from {ABLATION_CKPT_DIR / "emotic_context_only_best.pt"}")
else: 
    ctx_history = train_ablation_model(
        ctx_model, train_loader, val_loader, class_weights,
        tag="ContextOnly",
        ckpt_path=ABLATION_CKPT_DIR / "emotic_context_only_best.pt",
    )

### Evaluation

In [ ]:
ctx_metrics = evaluate_ablation_model(
    ctx_model, test_loader, EMOTIC_LABELS,
    tag="ContextOnly",
    save_prefix=ABLATION_CKPT_DIR / "emotic_context_only",
)

## 5. BodyOnly — entrenamiento y evaluación

ResNet50 ImageNet (backbone congelado) + cabeza lineal. Solo ve el recorte de cuerpo (bbox anotado).

In [ ]:
body_model = BodyOnlyModel(n_classes=len(EMOTIC_LABELS), freeze_backbone=True).to(device)
if (ABLATION_CKPT_DIR / "emotic_body_only_best.pt").exists():
    body_model.load_state_dict(torch.load(
        ABLATION_CKPT_DIR / "emotic_body_only_best.pt",
        map_location=device, weights_only=True,
    ))
    print(f"BodyOnly loaded from {ABLATION_CKPT_DIR / "emotic_body_only_best.pt"}")
else:
    body_history = train_ablation_model(
        body_model, train_loader, val_loader, class_weights,
        tag="BodyOnly",
        ckpt_path=ABLATION_CKPT_DIR / "emotic_body_only_best.pt",
    )

### Evaluation

In [ ]:
body_metrics = evaluate_ablation_model(
    body_model, test_loader, EMOTIC_LABELS,
    tag="BodyOnly",
    save_prefix=ABLATION_CKPT_DIR / "emotic_body_only",
)